In [0]:
from pyspark.sql import functions as F

In [0]:
events = spark.table("events_delta")

In [0]:
events.display()

In [0]:
from pyspark.sql import functions as F

features_df = events.groupBy("user_id").agg(
    F.count("*").alias("total_events"),
    F.count(F.when(F.col("event_type") == "purchase", 1)).alias("purchases"),
    F.sum("price").alias("total_spent"),
    F.avg("price").alias("avg_price")
)

In [0]:
features_df.display()

In [0]:
features_df = features_df.dropDuplicates(["user_id"])

In [0]:
features_df.groupBy("user_id").count().filter("count > 1").display()

In [0]:
features_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_user_features")

In [0]:
spark.sql("SELECT * FROM silver_user_features").display()

In [0]:
features_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in features_df.columns
]).display()

In [0]:
features_df.filter(F.col("total_spent") < 0).display()

In [0]:
features_df.describe().display()